# Davinci Notebook 8 — Efficient Attention & KV Cache

Reference implementation of SDPA-backed attention and incremental KV caching, including local-window eviction and global attention. This is a correctness-oriented prototype—not a production 500K-token kernel. Prefill still uses dense masks for local layers; benchmark and replace with optimized kernels before long-context training.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 1. Core primitives
SDPA uses optimized kernels when available. Boolean mask semantics in PyTorch SDPA: `True` means the position is allowed.

In [ ]:
def rope(x, positions, theta=10000.0):
    # x: [batch, heads, seq, head_dim]; split-half RoPE, even head dimension required
    d=x.shape[-1]; assert d%2==0
    inv=1.0/(theta**(torch.arange(0,d,2,device=x.device,dtype=torch.float32)/d))
    freqs=torch.outer(positions.to(torch.float32),inv)
    emb=torch.cat((freqs,freqs),dim=-1)
    cos,sin=emb.cos()[None,None,:,:],emb.sin()[None,None,:,:]
    a,b=x.chunk(2,dim=-1)
    return torch.cat((-b,a),dim=-1)*sin + x*cos

def local_causal_mask(q_positions, k_positions, window):
    # q_positions / k_positions are 1D absolute positions; [Q,K] mask
    delta=q_positions[:,None]-k_positions[None,:]
    return (delta>=0) & (delta<window)

class KVCache:
    def __init__(self, window=None): self.k=self.v=None; self.positions=None; self.window=window
    def append(self,k,v,positions):
        if self.k is None: self.k,self.v,self.positions=k,v,positions
        else:
            self.k=torch.cat((self.k,k),dim=2); self.v=torch.cat((self.v,v),dim=2); self.positions=torch.cat((self.positions,positions))
        if self.window is not None and self.positions.numel()>self.window:
            self.k=self.k[:,:, -self.window:,:]; self.v=self.v[:,:, -self.window:,:]; self.positions=self.positions[-self.window:]
        return self.k,self.v,self.positions


## 2. Attention module
Supports prefill and one-token incremental decode. Local layers retain only their window in the cache; global layers retain all cached tokens.

In [ ]:
class SDPAAttention(nn.Module):
    def __init__(self, hidden, heads, kv_heads, head_dim, window=None, theta=10000.):
        super().__init__(); assert heads%kv_heads==0 and head_dim%2==0
        self.heads,self.kv_heads,self.head_dim,self.window=heads,kv_heads,head_dim,window
        self.theta=theta; self.q=nn.Linear(hidden,heads*head_dim,bias=False); self.k=nn.Linear(hidden,kv_heads*head_dim,bias=False); self.v=nn.Linear(hidden,kv_heads*head_dim,bias=False); self.o=nn.Linear(heads*head_dim,hidden,bias=False)
    def forward(self,x,positions=None,cache=None):
        B,T,C=x.shape; device=x.device
        if positions is None: positions=torch.arange(T,device=device)
        q=self.q(x).view(B,T,self.heads,self.head_dim).transpose(1,2)
        k=self.k(x).view(B,T,self.kv_heads,self.head_dim).transpose(1,2); v=self.v(x).view(B,T,self.kv_heads,self.head_dim).transpose(1,2)
        q=rope(q,positions,self.theta); k=rope(k,positions,self.theta)
        if cache is not None: k,v,kpos=cache.append(k,v,positions)
        else: kpos=positions
        # Explicit GQA expansion for broad SDPA compatibility
        repeat=self.heads//self.kv_heads; k=k.repeat_interleave(repeat,dim=1); v=v.repeat_interleave(repeat,dim=1)
        if self.window is None and cache is None: mask=None; causal=True
        else:
            mask=local_causal_mask(positions,kpos,self.window if self.window is not None else int(kpos.numel())).to(device)
            mask=mask[None,None,:,:]; causal=False
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=causal,dropout_p=0.)
        return self.o(y.transpose(1,2).contiguous().view(B,T,-1))


## 3. Test parity: full forward vs incremental decode
Use a small random model and compare the logits for the same tokens. RoPE positions are absolute. This test exercises global and local attention separately.

In [ ]:
@torch.no_grad()
def parity_test(window=None):
    torch.manual_seed(123); device='cuda' if torch.cuda.is_available() else 'cpu'
    m=SDPAAttention(64,4,2,16,window=window).to(device).eval(); x=torch.randn(1,12,64,device=device); pos=torch.arange(12,device=device)
    full=m(x,pos); cache=KVCache(window=window); pieces=[]
    for i in range(x.size(1)):
        pieces.append(m(x[:,i:i+1,:],pos[i:i+1],cache=cache))
    decoded=torch.cat(pieces,dim=1); err=(full-decoded).abs().max().item()
    print('window=',window,'max abs error=',err); assert err<2e-4
parity_test(None); parity_test(5)


## 4. Cache memory estimates
Estimate assumes KV cache is stored in the indicated dtype, and counts only K and V tensors (not allocator overhead, metadata, temporary tensors, or model weights).

In [ ]:
def cache_bytes(batch, layers, global_layers, kv_heads, head_dim, tokens, local_window, bytes_per_element=2):
    total=0
    for is_global in global_layers:
        n=tokens if is_global else min(tokens,local_window)
        total += batch*2*n*kv_heads*head_dim*bytes_per_element
    return total

layers=28; global_flags=[(i+1)%4==0 for i in range(layers)]
for n in [4096,32768,65536,131072,262144,500000]:
    gib=cache_bytes(1,layers,global_flags,4,64,n,4096)/1024**3
    print(f'{n:,} tokens: {gib:.2f} GiB cache, batch=1, BF16')


## 5. Limitations and next steps
- This prototype's prefill local mask is dense and scales quadratically; it is not suitable for long-context prefill.
- Incremental decoding is one-token-at-a-time and not paged/continuous batching.
- Local cache eviction assumes monotonically increasing position IDs.
- Global layers retain the entire history and can dominate cache memory.
- Benchmark with the actual target GPU; replace prefill with FlashAttention-compatible sliding-window kernels and consider paged KV storage before 128K+ experiments.
- Validate parity and numerical stability across dtype, batch size, and hardware before integrating into the training pipeline.